In [ ]:
# ==============================================================
# R0 - Runtime / DuckDB lock-safe bootstrap
# ==============================================================
from pathlib import Path
import duckdb
import tomllib
import atexit

PROJECT_ROOT = Path.cwd()
CONFIG_TOML_PATH = PROJECT_ROOT / "benchmark_shared_config.toml"

if CONFIG_TOML_PATH.exists():
    with open(CONFIG_TOML_PATH, "rb") as f:
        SHARED_CONFIG = tomllib.load(f)
    print("Loaded shared config:", CONFIG_TOML_PATH)
else:
    SHARED_CONFIG = {
        "paths": {
            "data_dir": "content",
            "output_dir": "outputs_benchmark_survival",
            "tables_subdir": "tables",
            "metadata_subdir": "metadata",
            "data_output_subdir": "data",
            "duckdb_filename": "benchmark_survival.duckdb",
        },
        "benchmark": {
            "seed": 42,
            "test_size": 0.30,
            "early_window_weeks": 4,
            "main_enrollment_window_weeks": 4,
        },
    }
    print("Shared config TOML not found. Using defaults in-memory.")

paths_cfg = SHARED_CONFIG.get("paths", {})
SHARED_BENCHMARK_CONFIG = SHARED_CONFIG.get("benchmark", {})

def _resolve_project_path(raw_path: str) -> Path:
    p = Path(raw_path)
    return p if p.is_absolute() else PROJECT_ROOT / p

DATA_DIR = _resolve_project_path(paths_cfg.get("data_dir", "content"))
OUTPUT_DIR = _resolve_project_path(paths_cfg.get("output_dir", "outputs_benchmark_survival"))
TABLES_DIR = OUTPUT_DIR / paths_cfg.get("tables_subdir", "tables")
METADATA_DIR = OUTPUT_DIR / paths_cfg.get("metadata_subdir", "metadata")
DATA_OUTPUT_DIR = OUTPUT_DIR / paths_cfg.get("data_output_subdir", "data")
DUCKDB_PATH = OUTPUT_DIR / paths_cfg.get("duckdb_filename", "benchmark_survival.duckdb")

for p in [OUTPUT_DIR, TABLES_DIR, METADATA_DIR, DATA_OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

if "con" in globals():
    try:
        con.close()
        print("Closed previous DuckDB connection before reconnecting.")
    except Exception as _close_err:
        print(f"Warning while closing previous DuckDB connection: {_close_err}")

con = duckdb.connect(str(DUCKDB_PATH))

def _close_duckdb_connection() -> None:
    if "con" in globals():
        try:
            con.close()
            print("DuckDB connection closed.")
        except Exception:
            pass

if "_duckdb_close_registered" not in globals():
    atexit.register(_close_duckdb_connection)
    _duckdb_close_registered = True

print("Runtime context ready.")
print("- OUTPUT_DIR  :", OUTPUT_DIR)
print("- DUCKDB_PATH :", DUCKDB_PATH)

# E — Post-hoc Audits and Dependency Map

This section keeps only the late-stage audit blocks that still matter for the manuscript pipeline or for reviewer-facing robustness checks.

Use the dependency map below before rerunning isolated cells. The paper-facing path is now: benchmark outputs -> calibration/ablation/explainability audits -> curated paper freeze.

In [ ]:
# ==============================================================
# E0 — Dependency Map for Late-Stage Audit Sections
# --------------------------------------------------------------
# Purpose:
#   Make late-stage dependencies explicit before the notebook is
#   rerun selectively after the editorial cleanup.
#
# Methodological note:
#   This cell documents execution contracts only.
#   It does not train models and does not recompute metrics.
# ==============================================================

print("\n" + "=" * 70)
print("E0 — Dependency Map for Late-Stage Audit Sections")
print("=" * 70)
print("Methodological note: this cell documents dependencies only.")

required_names = ["TABLES_DIR", "METADATA_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import pandas as pd

dependency_rows = [
    {
        "section_id": "E1",
        "title": "Benchmark comparability audit",
        "depends_on": "canonical tuned benchmark tables",
        "primary_outputs": "table_p10_0_benchmark_comparability_*",
        "paper_facing": False,
        "notes": "Standalone audit retained as optional post-hoc check."
    },
    {
        "section_id": "E2",
        "title": "Calibration artifact audit",
        "depends_on": "per-model calibration tables + benchmark-wide calibration tables",
        "primary_outputs": "table_p11_0_calibration_*",
        "paper_facing": True,
        "notes": "Feeds calibration strengthening and final paper freeze."
    },
    {
        "section_id": "E3",
        "title": "Calibration slope/intercept audit",
        "depends_on": "E2 calibration artifacts or legacy reliability bins",
        "primary_outputs": "table_p11_1_calibration_slope_intercept_*",
        "paper_facing": True,
        "notes": "Optional strengthening layer merged into calibration summaries when available."
    },
    {
        "section_id": "E4",
        "title": "Unified calibration strengthening summary",
        "depends_on": "E2 + E3",
        "primary_outputs": "table_p11_2_calibration_strengthening_*",
        "paper_facing": True,
        "notes": "Strengthens reviewer-facing calibration interpretation."
    },
    {
        "section_id": "E5",
        "title": "Sensitivity artifact audit",
        "depends_on": "sensitivity outputs if that branch is executed",
        "primary_outputs": "table_p12_0_sensitivity_*",
        "paper_facing": False,
        "notes": "Kept as optional post-hoc robustness layer."
    },
    {
        "section_id": "E6",
        "title": "Horizon-choice stress test",
        "depends_on": "E5",
        "primary_outputs": "table_p12_3_horizon_choice_*",
        "paper_facing": False,
        "notes": "Optional; not required by the manuscript freeze."
    },
    {
        "section_id": "E7",
        "title": "Unified sensitivity summary",
        "depends_on": "E5 + E6",
        "primary_outputs": "table_p12_4_sensitivity_*",
        "paper_facing": False,
        "notes": "Optional; retained for robustness documentation only."
    },
    {
        "section_id": "F1",
        "title": "Define ablation protocol",
        "depends_on": "benchmark-ready tuned datasets",
        "primary_outputs": "table_ablation_model_registry.csv and companions",
        "paper_facing": True,
        "notes": "Foundational registry for all downstream ablation cells."
    },
    {
        "section_id": "F2",
        "title": "Materialize ablation variants",
        "depends_on": "F1 + tuned ready datasets",
        "primary_outputs": "table_ablation_variant_registry.csv",
        "paper_facing": True,
        "notes": "Feeds both discrete-time and continuous-time ablation training cells."
    },
    {
        "section_id": "F3",
        "title": "Discrete-time tuned ablation runs",
        "depends_on": "F2",
        "primary_outputs": "table_ablation_discrete_*",
        "paper_facing": True,
        "notes": "Feeds F5 consolidated ablation outputs."
    },
    {
        "section_id": "F4",
        "title": "Continuous-time tuned ablation runs",
        "depends_on": "F2",
        "primary_outputs": "table_ablation_continuous_*",
        "paper_facing": True,
        "notes": "Feeds F5 consolidated ablation outputs."
    },
    {
        "section_id": "F5",
        "title": "Consolidate ablation results",
        "depends_on": "F3 + F4",
        "primary_outputs": "table_ablation_summary_by_model.csv",
        "paper_facing": True,
        "notes": "Canonical source for the paper ablation table and figure."
    },
    {
        "section_id": "F6",
        "title": "Preprocessing and tuning audit",
        "depends_on": "tuning result tables + preprocessing summaries",
        "primary_outputs": "table_preprocessing_and_tuning_audit.csv",
        "paper_facing": True,
        "notes": "Feeds appendix preprocessing/tuning table."
    },
    {
        "section_id": "F7",
        "title": "Bootstrap uncertainty audit",
        "depends_on": "saved tuned models + test-ready datasets",
        "primary_outputs": "table_appendix_bootstrap_uncertainty_compact.csv",
        "paper_facing": True,
        "notes": "Feeds appendix uncertainty table."
    },
    {
        "section_id": "F8",
        "title": "Comparable Cox PH audit",
        "depends_on": "cox tuned model + enrollment_cox_ready_train",
        "primary_outputs": "table_appendix_cox_ph_global_summary.csv + fig_appendix_cox_ph_diagnostics.png",
        "paper_facing": True,
        "notes": "Feeds appendix PH table and figure."
    },
    {
        "section_id": "G1",
        "title": "Define explainability protocol",
        "depends_on": "tuned benchmark families already frozen",
        "primary_outputs": "table_explainability_model_registry.csv",
        "paper_facing": True,
        "notes": "Registry cell for the full explainability layer."
    },
    {
        "section_id": "G2",
        "title": "Linear tuned explainability",
        "depends_on": "linear tuned model + preprocessor",
        "primary_outputs": "table_linear_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G3",
        "title": "Neural tuned explainability",
        "depends_on": "neural tuned ready data + preprocessor",
        "primary_outputs": "table_neural_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G4",
        "title": "Cox tuned explainability",
        "depends_on": "cox tuned model + preprocessor",
        "primary_outputs": "table_cox_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G5",
        "title": "DeepSurv tuned explainability",
        "depends_on": "deepsurv tuned ready data + preprocessor",
        "primary_outputs": "table_deepsurv_explainability_*",
        "paper_facing": True,
        "notes": "Feeds G6 consolidated explainability outputs."
    },
    {
        "section_id": "G6",
        "title": "Consolidate explainability",
        "depends_on": "G2 + G3 + G4 + G5",
        "primary_outputs": "table_explainability_summary_by_model.csv + table_explainability_all_blocks_normalized.csv",
        "paper_facing": True,
        "notes": "Canonical source for the paper explainability table and figure."
    },
    {
        "section_id": "G7",
        "title": "Freeze curated paper artifacts",
        "depends_on": "benchmark leaderboard + E2/E3/E4 + F5/F6/F7/F8 + G6",
        "primary_outputs": "outputs_benchmark_survival/paper_main and paper_appendix",
        "paper_facing": True,
        "notes": "Curates only the CSV and PNG artifacts cited by the manuscript."
    },
    {
        "section_id": "G8",
        "title": "Display curated paper figures",
        "depends_on": "G7",
        "primary_outputs": "notebook displays only",
        "paper_facing": True,
        "notes": "Visual QA for exported PNG assets."
    },
    {
        "section_id": "G9",
        "title": "Preview curated paper evidence",
        "depends_on": "G7",
        "primary_outputs": "notebook prints and displays only",
        "paper_facing": True,
        "notes": "Notebook-side synthesis from curated paper artifacts only."
    },
]

dependency_map_df = pd.DataFrame(dependency_rows)
dependency_map_path = TABLES_DIR / "table_late_stage_dependency_map.csv"
dependency_map_df.to_csv(dependency_map_path, index=False)

paper_path_df = dependency_map_df[dependency_map_df["paper_facing"]].copy()

print("\nLate-stage dependency map:")
display(dependency_map_df)

print("\nPaper-facing execution path:")
display(paper_path_df[["section_id", "title", "depends_on", "primary_outputs"]])

print("\nCurated late-stage policy:")
print("- Auxiliary comparable discrete arm cells were removed from the notebook.")
print("- Narrative interpretation markdown cells were removed so conclusions come from executed outputs.")
print("- The curated manuscript path now terminates at outputs_benchmark_survival/paper_main and outputs_benchmark_survival/paper_appendix.")

print("\nSaved:")
print("-", dependency_map_path.resolve())

# E1 — Benchmark Comparability Audit

### Funcao: add_manual_row

Definicao isolada de add_manual_row para reutilizacao nas etapas seguintes.


In [ ]:
# ============================================================
# E1 — Benchmark Comparability Audit
# ============================================================
# Purpose:
#   Audit structural comparability across benchmark model families.
#   This step does not retrain anything. It creates an auditable map
#   of which inputs/tables/representations are used by each family.
#
# Main outputs:
#   - table_p10_0_benchmark_comparability_audit.csv
#   - table_p10_0_benchmark_comparability_summary.csv
#   - metadata_p10_0_benchmark_comparability_audit.json
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E1 — Benchmark Comparability Audit")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

available_tables = sorted(
    con.execute("SHOW TABLES").fetchdf()["name"].astype(str).tolist()
)

# ------------------------------------------------------------
# 1) Candidate benchmark-related tables
# ------------------------------------------------------------
candidate_rows = []

for tbl in available_tables:
    lname = tbl.lower()

    is_relevant = any(token in lname for token in [
        "pp_", "person_period", "hazard", "survival",
        "cox", "deepsurv", "discrete", "continuous",
        "neural", "linear", "input", "ready", "split"
    ])

    if not is_relevant:
        continue

    try:
        cols_df = con.execute(f"PRAGMA table_info('{tbl}')").fetchdf()
        cols = cols_df["name"].astype(str).tolist()
        n_rows = con.execute(f"SELECT COUNT(*) AS n FROM {tbl}").fetchdf()["n"].iloc[0]

        colset = set(cols)
        has_week = "week" in colset
        has_id_student = "id_student" in colset
        has_module = "code_module" in colset
        has_presentation = "code_presentation" in colset
        has_event = "event_observed" in colset or "event" in colset
        has_t_final = "t_final_week" in colset
        has_t_event = "t_event_week" in colset

        # crude representation heuristics
        if has_week and n_rows > 50000:
            temporal_representation = "dynamic_weekly"
        elif (not has_week) and has_t_final:
            temporal_representation = "enrollment_level_survival"
        elif (not has_week) and ("week" not in colset):
            temporal_representation = "static_or_summary_level"
        else:
            temporal_representation = "other"

        # family heuristics from table name
        if "cox" in lname:
            benchmark_family = "continuous_time"
        elif "deepsurv" in lname:
            benchmark_family = "continuous_time_neural"
        elif "linear_hazard" in lname or "neural_hazard" in lname or "hazard" in lname:
            benchmark_family = "discrete_time"
        elif "person_period" in lname or "pp_" in lname:
            benchmark_family = "person_period_upstream"
        elif "survival" in lname:
            benchmark_family = "enrollment_survival_upstream"
        else:
            benchmark_family = "other"

        candidate_rows.append({
            "table_name": tbl,
            "n_rows": int(n_rows),
            "n_columns": int(len(cols)),
            "has_id_student": has_id_student,
            "has_code_module": has_module,
            "has_code_presentation": has_presentation,
            "has_week": has_week,
            "has_event_signal": has_event,
            "has_t_event_week": has_t_event,
            "has_t_final_week": has_t_final,
            "temporal_representation_detected": temporal_representation,
            "benchmark_family_detected": benchmark_family,
            "columns_preview": ", ".join(cols[:20]),
        })

    except Exception as e:
        candidate_rows.append({
            "table_name": tbl,
            "n_rows": np.nan,
            "n_columns": np.nan,
            "has_id_student": np.nan,
            "has_code_module": np.nan,
            "has_code_presentation": np.nan,
            "has_week": np.nan,
            "has_event_signal": np.nan,
            "has_t_event_week": np.nan,
            "has_t_final_week": np.nan,
            "temporal_representation_detected": "error",
            "benchmark_family_detected": "error",
            "columns_preview": f"ERROR: {str(e)}",
        })

table_candidates = pd.DataFrame(candidate_rows)

if table_candidates.empty:
    raise RuntimeError(
        "No benchmark-related tables were detected. "
        "Run the benchmark notebook steps before P10.0."
    )

# ------------------------------------------------------------
# 2) Manual structural audit layer
# ------------------------------------------------------------
# This layer translates the current benchmark architecture into a
# model-family comparability map. It is intentionally explicit.
#
# You may edit labels later if the notebook architecture evolves.

manual_rows = []

def add_manual_row(
    model_label,
    family,
    likely_input_table,
    representation_type,
    update_regime,
    benchmark_role,
    comparability_status,
    rationale
):
    manual_rows.append({
        "model_or_family_label": model_label,
        "family": family,
        "likely_input_table": likely_input_table,
        "representation_type": representation_type,
        "update_regime": update_regime,
        "benchmark_role": benchmark_role,
        "comparability_status": comparability_status,
        "rationale": rationale,
    })

# Upstream enrollment-level table
add_manual_row(
    model_label="Enrollment survival backbone",
    family="upstream_enrollment_level",
    likely_input_table="enrollment_survival_ready",
    representation_type="enrollment_level_survival",
    update_regime="not_a_model",
    benchmark_role="upstream_backbone",
    comparability_status="not_applicable",
    rationale="Canonical enrollment-level event/time table used upstream."
)

# Person-period upstream tables
for tbl_name in ["person_period_min", "person_period_enriched", "pp_linear_hazard_input", "pp_neural_hazard_input"]:
    if tbl_name in table_candidates["table_name"].values:
        add_manual_row(
            model_label=tbl_name,
            family="discrete_time_upstream",
            likely_input_table=tbl_name,
            representation_type="dynamic_weekly",
            update_regime="weekly_updating",
            benchmark_role="upstream_discrete_time_input",
            comparability_status="supports_dynamic_regime",
            rationale="Person-period structure with week-level rows enables dynamic weekly updating."
        )

# Discrete-time linear hazard
if "pp_linear_hazard_ready_train" in table_candidates["table_name"].values or "pp_linear_hazard_input" in table_candidates["table_name"].values:
    add_manual_row(
        model_label="Discrete-time linear hazard",
        family="discrete_time",
        likely_input_table="pp_linear_hazard_*",
        representation_type="dynamic_weekly",
        update_regime="weekly_updating",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_static_continuous",
        rationale="Uses person-period rows and week-indexed dynamic updating."
    )

# Discrete-time neural hazard
if "pp_neural_hazard_ready_train" in table_candidates["table_name"].values or "pp_neural_hazard_input" in table_candidates["table_name"].values:
    add_manual_row(
        model_label="Discrete-time neural hazard",
        family="discrete_time_neural",
        likely_input_table="pp_neural_hazard_*",
        representation_type="dynamic_weekly",
        update_regime="weekly_updating",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_static_continuous",
        rationale="Uses person-period rows and week-indexed dynamic updating."
    )

# Continuous-time Cox comparable arm
cox_like_tables = [x for x in table_candidates["table_name"].tolist() if "cox" in x.lower()]
if len(cox_like_tables) > 0:
    add_manual_row(
        model_label="Comparable Cox-based continuous-time arm",
        family="continuous_time",
        likely_input_table=" / ".join(cox_like_tables[:5]),
        representation_type="early_window_summary_or_enrollment_level",
        update_regime="static_after_early_window",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_dynamic_discrete",
        rationale="Continuous-time comparable models are represented at enrollment/static summary level rather than dynamic weekly person-period updating."
    )

# Deepsurv comparable arm
deepsurv_like_tables = [x for x in table_candidates["table_name"].tolist() if "deepsurv" in x.lower()]
if len(deepsurv_like_tables) > 0:
    add_manual_row(
        model_label="Comparable DeepSurv arm",
        family="continuous_time_neural",
        likely_input_table=" / ".join(deepsurv_like_tables[:5]),
        representation_type="early_window_summary_or_enrollment_level",
        update_regime="static_after_early_window",
        benchmark_role="main_benchmark_model",
        comparability_status="structurally_asymmetric_vs_dynamic_discrete",
        rationale="Neural continuous-time arm appears to rely on enrollment-level summarized inputs rather than dynamic weekly rows."
    )

# Context-held-out split is not a family, but relevant to benchmark design
if "enrollment_survival_ready_context_split" in available_tables:
    add_manual_row(
        model_label="Context-held-out split backbone",
        family="evaluation_protocol",
        likely_input_table="enrollment_survival_ready_context_split",
        representation_type="enrollment_level_split_protocol",
        update_regime="not_a_model",
        benchmark_role="alternative_transportability_protocol",
        comparability_status="not_applicable",
        rationale="Alternative split protocol for unseen curricular contexts; does not itself change family comparability."
    )

table_p10_0_benchmark_comparability_audit = pd.DataFrame(manual_rows)

# ------------------------------------------------------------
# 3) Family-level summary
# ------------------------------------------------------------
summary = (
    table_p10_0_benchmark_comparability_audit
    .groupby(["family", "representation_type", "update_regime", "comparability_status"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .sort_values(["family", "representation_type", "update_regime"])
    .reset_index(drop=True)
)

# overall interpretive summary
overall_findings = []

n_dynamic = int(
    (table_p10_0_benchmark_comparability_audit["representation_type"] == "dynamic_weekly").sum()
)
n_static_summary = int(
    (table_p10_0_benchmark_comparability_audit["representation_type"] == "early_window_summary_or_enrollment_level").sum()
)
n_asym = int(
    table_p10_0_benchmark_comparability_audit["comparability_status"]
    .astype(str)
    .str.contains("asymmetric", case=False, na=False)
    .sum()
)

overall_findings.append({
    "finding": "n_dynamic_weekly_entries",
    "value": n_dynamic,
})
overall_findings.append({
    "finding": "n_early_window_or_enrollment_level_entries",
    "value": n_static_summary,
})
overall_findings.append({
    "finding": "n_entries_flagged_as_structurally_asymmetric",
    "value": n_asym,
})
overall_findings.append({
    "finding": "benchmark_comparability_diagnosis",
    "value": (
        "Current benchmark includes both dynamic weekly discrete-time arms and "
        "static/early-window continuous-time comparable arms; therefore family-level "
        "ranking should not be interpreted as a pure architecture-only comparison."
    )
})

table_p10_0_benchmark_comparability_summary = pd.DataFrame(overall_findings)

# ------------------------------------------------------------
# 4) Save outputs
# ------------------------------------------------------------
path_candidates = OUT_TABLES / "table_p10_0_detected_benchmark_related_tables.csv"
path_audit = OUT_TABLES / "table_p10_0_benchmark_comparability_audit.csv"
path_summary = OUT_TABLES / "table_p10_0_benchmark_comparability_summary.csv"
path_metadata = OUT_METADATA / "metadata_p10_0_benchmark_comparability_audit.json"

table_candidates.to_csv(path_candidates, index=False)
table_p10_0_benchmark_comparability_audit.to_csv(path_audit, index=False)
table_p10_0_benchmark_comparability_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P10.0",
    "title": "Benchmark Comparability Audit",
    "purpose": "Audit structural comparability across benchmark families before redesign.",
    "main_diagnosis": (
        "The current benchmark mixes dynamic weekly discrete-time inputs with "
        "static or early-window continuous-time comparable inputs."
    ),
    "output_tables": [
        path_candidates.as_posix(),
        path_audit.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 5) Display
# ------------------------------------------------------------
print("\nDetected benchmark-related tables:")
display(table_candidates)

print("\nComparability audit:")
display(table_p10_0_benchmark_comparability_audit)

print("\nComparability summary:")
display(table_p10_0_benchmark_comparability_summary)

print("\nSaved:")
print("-", path_candidates.resolve())
print("-", path_audit.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())

# P10.5 — Integrate Comparable Discrete-Time Minimal Arm into

In [ ]:
# ==============================================================
# P10.5 — Integrate Comparable Discrete-Time Minimal Arm into
#          Benchmark Results
# ==============================================================
# Purpose:
#   Consolidate benchmark results for the comparable arm,
#   incorporating the newly evaluated
#   discrete_time_comparable_minimal branch.
#
# Main outputs:
#   - table_p10_5_comparable_arm_benchmark_results.csv
#   - table_p10_5_comparable_arm_best_by_variant.csv
#   - table_p10_5_benchmark_result_inventory.csv
#   - metadata_p10_5_comparable_arm_integration.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("P10.5 — Integrate Comparable Discrete-Time Minimal Arm into Benchmark Results")
print("=" * 70)

required_names = ["con", "MODEL_INPUT_VARIANTS"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Run prior setup cells first."
    )

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Load currently available result artifacts
# --------------------------------------------------------------
path_new_discrete = OUT_TABLES / "table_p10_4_comparable_discrete_metrics_by_horizon.csv"
if not path_new_discrete.exists():
    raise FileNotFoundError(
        f"Required file not found: {path_new_discrete.resolve()}. Run P10.4 first."
    )

new_discrete_df = pd.read_csv(path_new_discrete)

# Optional existing comparable-arm result sources
candidate_existing_files = [
    OUT_TABLES / "table_enrollment_model_metrics.csv",
    OUT_TABLES / "table_benchmark_enrollment_model_metrics.csv",
    OUT_TABLES / "table_main_benchmark_metrics.csv",
    OUT_TABLES / "table_model_metrics.csv",
    OUT_TABLES / "table_benchmark_results.csv",
]

existing_loaded = []
for p in candidate_existing_files:
    if p.exists():
        try:
            df = pd.read_csv(p)
            df["source_file"] = p.name
            existing_loaded.append(df)
        except Exception:
            pass

# --------------------------------------------------------------
# 2) Normalize the new comparable discrete result table
# --------------------------------------------------------------
disc = new_discrete_df.copy()

# Normalize column names to a benchmark-friendly schema
disc_norm = pd.DataFrame({
    "benchmark_variant": disc["benchmark_variant"],
    "benchmark_arm": disc["benchmark_arm"],
    "model_family": disc["benchmark_variant"],
    "model_label": "Comparable discrete-time minimal arm",
    "representation_type": disc["representation_type"],
    "update_regime": disc["update_regime"],
    "model_class": disc["model_class"],
    "horizon": pd.to_numeric(disc["horizon"], errors="coerce"),
    "metric_primary_name": "auc_enrollment_score_by_horizon",
    "metric_primary_value": pd.to_numeric(disc["auc_enrollment_score_by_horizon"], errors="coerce"),
    "metric_ap_name": "average_precision_by_horizon",
    "metric_ap_value": pd.to_numeric(disc["average_precision_by_horizon"], errors="coerce"),
    "metric_brier_name": "brier_by_horizon_on_risk",
    "metric_brier_value": pd.to_numeric(disc["brier_by_horizon_on_risk"], errors="coerce"),
    "n_test_enrollments_total": pd.to_numeric(disc["n_test_enrollments_total"], errors="coerce"),
    "n_test_enrollments_with_support": pd.to_numeric(disc["n_test_enrollments_with_support"], errors="coerce"),
    "support_fraction": pd.to_numeric(disc["support_fraction"], errors="coerce"),
    "positive_rate_by_horizon_supported": pd.to_numeric(disc["positive_rate_by_horizon_supported"], errors="coerce"),
    "feature_set_used": disc["feature_set_used"].astype(str),
    "result_status": "materialized_now",
    "source_note": "P10.4 comparable discrete-time minimal arm",
})

# --------------------------------------------------------------
# 3) Build inventory of currently known benchmark-result files
# --------------------------------------------------------------
inventory_rows = [{
    "artifact_name": path_new_discrete.name,
    "artifact_exists": True,
    "artifact_role": "new_comparable_discrete_result",
    "n_rows": int(len(new_discrete_df)),
}]

for p in candidate_existing_files:
    inventory_rows.append({
        "artifact_name": p.name,
        "artifact_exists": p.exists(),
        "artifact_role": "optional_existing_benchmark_result_source",
        "n_rows": int(len(pd.read_csv(p))) if p.exists() else np.nan,
    })

table_p10_5_benchmark_result_inventory = pd.DataFrame(inventory_rows)

# --------------------------------------------------------------
# 4) Attempt soft integration with existing comparable-arm results
# --------------------------------------------------------------
# Since historical metric tables may vary in schema, do not force a brittle merge.
# Instead, keep the new result canonical and attach any existing sources as inventory.
#
# This table is the new canonical comparable-arm result table for downstream use.
table_p10_5_comparable_arm_benchmark_results = disc_norm.copy()

# Best row by variant according to primary metric (AUC)
best_idx = (
    table_p10_5_comparable_arm_benchmark_results
    .groupby("benchmark_variant")["metric_primary_value"]
    .idxmax()
    .dropna()
    .astype(int)
    .tolist()
)

table_p10_5_comparable_arm_best_by_variant = (
    table_p10_5_comparable_arm_benchmark_results
    .loc[best_idx]
    .sort_values(["metric_primary_value", "horizon"], ascending=[False, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Persist a DuckDB table for downstream use
# --------------------------------------------------------------
con.register(
    "tmp_p10_5_comparable_arm_benchmark_results_df",
    table_p10_5_comparable_arm_benchmark_results
)
con.execute("DROP TABLE IF EXISTS comparable_arm_benchmark_results")
con.execute("""
    CREATE TABLE comparable_arm_benchmark_results AS
    SELECT *
    FROM tmp_p10_5_comparable_arm_benchmark_results_df
""")

# --------------------------------------------------------------
# 6) Save outputs
# --------------------------------------------------------------
path_results = OUT_TABLES / "table_p10_5_comparable_arm_benchmark_results.csv"
path_best = OUT_TABLES / "table_p10_5_comparable_arm_best_by_variant.csv"
path_inventory = OUT_TABLES / "table_p10_5_benchmark_result_inventory.csv"
path_metadata = OUT_METADATA / "metadata_p10_5_comparable_arm_integration.json"

table_p10_5_comparable_arm_benchmark_results.to_csv(path_results, index=False)
table_p10_5_comparable_arm_best_by_variant.to_csv(path_best, index=False)
table_p10_5_benchmark_result_inventory.to_csv(path_inventory, index=False)

metadata = {
    "step": "P10.5",
    "title": "Integrate Comparable Discrete-Time Minimal Arm into Benchmark Results",
    "new_variant_integrated": "discrete_time_comparable_minimal",
    "output_duckdb_table": "comparable_arm_benchmark_results",
    "notes": [
        "This step consolidates the comparable discrete-time minimal arm into the benchmark result structure.",
        "It does not retrain older branches.",
        "The result table created here is the canonical comparable-arm result table for downstream use.",
        "If historical comparable-arm metric tables exist in alternative schemas, they are tracked in the inventory rather than brittle-merged automatically."
    ],
    "output_tables": [
        path_results.as_posix(),
        path_best.as_posix(),
        path_inventory.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 7) Display
# --------------------------------------------------------------
print("\nBenchmark result inventory:")
display(table_p10_5_benchmark_result_inventory)

print("\nComparable-arm benchmark results:")
display(table_p10_5_comparable_arm_benchmark_results)

print("\nBest result by variant:")
display(table_p10_5_comparable_arm_best_by_variant)

print("\nSaved:")
print("-", path_results.resolve())
print("-", path_best.resolve())
print("-", path_inventory.resolve())
print("-", path_metadata.resolve())

# E2 — Survival Calibration Audit

In [ ]:
# ==============================================================
# E2 — Survival Calibration Audit
# CORRECTED VERSION ALIGNED WITH CURRENT NOTEBOOK CONVENTION
# ==============================================================
# Purpose:
#   Audit currently available calibration evidence in the notebook
#   and summarize what already exists vs. what is still missing,
#   using the current artifact naming convention.
#
# Main outputs:
#   - table_p11_0_calibration_artifact_inventory.csv
#   - table_p11_0_calibration_audit_summary.csv
#   - table_p11_0_calibration_missing_components.csv
#   - metadata_p11_0_survival_calibration_audit.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E2 — Survival Calibration Audit")
print("=" * 70)

if "con" not in globals():
    raise NameError("DuckDB connection 'con' not found. Run the setup cells first.")

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_FIGURES = OUT_BASE / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Inventory calibration-related files under current convention
# --------------------------------------------------------------
candidate_files = []

for folder, kind in [
    (OUT_TABLES, "table"),
    (OUT_FIGURES, "figure"),
    (OUT_METADATA, "metadata"),
]:
    if folder.exists():
        for p in sorted(folder.iterdir()):
            name_l = p.name.lower()
            if any(token in name_l for token in [
                "calibration",
                "reliability",
                "brier",
                "support_by_horizon",
                "by_horizon",
                "paper_summary",
                "audit_tuned_models",
                "benchmark_calibration",
            ]):
                candidate_files.append({
                    "artifact_name": p.name,
                    "artifact_path": str(p.resolve()),
                    "artifact_type": kind,
                    "exists": True,
                    "size_bytes": p.stat().st_size if p.exists() else np.nan,
                })

table_p11_0_calibration_artifact_inventory = pd.DataFrame(candidate_files)

if table_p11_0_calibration_artifact_inventory.empty:
    table_p11_0_calibration_artifact_inventory = pd.DataFrame([{
        "artifact_name": "(none found)",
        "artifact_path": "",
        "artifact_type": "none",
        "exists": False,
        "size_bytes": np.nan,
    }])

# --------------------------------------------------------------
# 2) Detect currently available calibration evidence classes
# --------------------------------------------------------------
table_names = set(
    table_p11_0_calibration_artifact_inventory.loc[
        table_p11_0_calibration_artifact_inventory["artifact_type"] == "table",
        "artifact_name"
    ].astype(str).tolist()
)


### Funcao: any_match

Definicao isolada de any_match para reutilizacao nas etapas seguintes.


In [ ]:

def any_match(prefixes_or_names):
    return any(name in table_names for name in prefixes_or_names)


### Funcao: any_contains

Definicao isolada de any_contains para reutilizacao nas etapas seguintes.


In [ ]:

def any_contains(substrs):
    return any(any(sub in name for sub in substrs) for name in table_names)


In [ ]:

has_model_level_bins = any_contains(["_calibration_bins_by_horizon.csv"])
has_model_level_summary = any_contains(["_calibration_summary.csv"])
has_model_level_brier = any_contains(["_brier_by_horizon.csv"])
has_model_level_support = any_contains(["_support_by_horizon.csv"])
has_benchmark_wide_cal = "table_benchmark_calibration_by_horizon_wide.csv" in table_names
has_benchmark_wide_brier = "table_benchmark_brier_by_horizon_wide.csv" in table_names
has_benchmark_wide_auc = "table_benchmark_risk_auc_by_horizon_wide.csv" in table_names
has_all_tuned_audit = "table_calibration_audit_all_tuned_models.csv" in table_names
has_paper_summary = "table_calibration_paper_summary_all_tuned_models.csv" in table_names
has_reliability_bins_all_tuned = "table_calibration_reliability_bins_all_tuned_models.csv" in table_names
has_reliability_bins_h20 = "table_calibration_reliability_bins_h20_all_tuned_models.csv" in table_names

# --------------------------------------------------------------
# 3) Build corrected audit summary
# --------------------------------------------------------------
summary_rows = [
    {
        "evidence_group": "model_level_calibration_bins_by_horizon",
        "available": has_model_level_bins,
        "notes": "Per-model calibration bins by horizon detected under current naming convention."
    },
    {
        "evidence_group": "model_level_calibration_summary",
        "available": has_model_level_summary,
        "notes": "Per-model calibration summary tables detected under current naming convention."
    },
    {
        "evidence_group": "model_level_brier_by_horizon",
        "available": has_model_level_brier,
        "notes": "Per-model Brier-by-horizon tables detected."
    },
    {
        "evidence_group": "model_level_support_by_horizon",
        "available": has_model_level_support,
        "notes": "Per-model support-by-horizon tables detected."
    },
    {
        "evidence_group": "benchmark_wide_calibration_table",
        "available": has_benchmark_wide_cal,
        "notes": "Wide benchmark calibration table by horizon detected."
    },
    {
        "evidence_group": "benchmark_wide_brier_table",
        "available": has_benchmark_wide_brier,
        "notes": "Wide benchmark Brier-by-horizon table detected."
    },
    {
        "evidence_group": "benchmark_wide_auc_table",
        "available": has_benchmark_wide_auc,
        "notes": "Wide benchmark risk-AUC-by-horizon table detected."
    },
    {
        "evidence_group": "all_tuned_calibration_audit",
        "available": has_all_tuned_audit,
        "notes": "Calibration audit across tuned models detected."
    },
    {
        "evidence_group": "paper_summary_all_tuned_models",
        "available": has_paper_summary,
        "notes": "Paper-facing calibration summary across tuned models detected."
    },
    {
        "evidence_group": "reliability_bins_all_tuned_models",
        "available": has_reliability_bins_all_tuned,
        "notes": "Reliability bins across tuned models detected."
    },
    {
        "evidence_group": "reliability_bins_h20_all_tuned_models",
        "available": has_reliability_bins_h20,
        "notes": "Reliability bins at horizon 20 across tuned models detected."
    },
]

table_p11_0_calibration_audit_summary = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 4) Missing / stronger components audit
# --------------------------------------------------------------
missing_components = [
    {
        "component": "horizon_specific_calibration_summary",
        "status": "already_present",
        "why_it_matters": "Reviewer requested stronger survival calibration evidence.",
        "recommended_next_step": "Reuse and cite existing horizon-specific calibration tables.",
        "current_evidence_note": "Model-level and benchmark-wide horizon-specific calibration artifacts already exist."
    },
    {
        "component": "benchmark_calibration_comparison_table",
        "status": "already_present_but_may_need_final_paper_framing",
        "why_it_matters": "Needed for side-by-side reporting in the paper.",
        "recommended_next_step": "Map existing benchmark-wide calibration table to paper-ready narrative/table.",
        "current_evidence_note": "table_benchmark_calibration_by_horizon_wide.csv already exists."
    },
    {
        "component": "calibration_slope_and_intercept_by_horizon",
        "status": "still_missing",
        "why_it_matters": "Strengthens calibration evaluation beyond binned summaries.",
        "recommended_next_step": "Estimate horizon-specific slope/intercept diagnostics.",
        "current_evidence_note": "No canonical slope/intercept artifact detected."
    },
    {
        "component": "integrated_calibration_style_index",
        "status": "still_missing",
        "why_it_matters": "Provides a stronger integrated calibration summary than raw bin tables alone.",
        "recommended_next_step": "Implement horizon-wise ICI-style summary or nearest operational equivalent.",
        "current_evidence_note": "No canonical ICI-style artifact detected."
    },
    {
        "component": "explicit_calibration_strengthening_for_reviewer_response",
        "status": "partially_missing",
        "why_it_matters": "Reviewer specifically questioned robustness of calibration assessment.",
        "recommended_next_step": "Add at least one stronger calibration diagnostic beyond bin summaries.",
        "current_evidence_note": "Rich calibration evidence exists already, but stronger diagnostics are still needed."
    },
]

table_p11_0_calibration_missing_components = pd.DataFrame(missing_components)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p11_0_calibration_artifact_inventory.csv"
path_summary = OUT_TABLES / "table_p11_0_calibration_audit_summary.csv"
path_missing = OUT_TABLES / "table_p11_0_calibration_missing_components.csv"
path_metadata = OUT_METADATA / "metadata_p11_0_survival_calibration_audit.json"

table_p11_0_calibration_artifact_inventory.to_csv(path_inventory, index=False)
table_p11_0_calibration_audit_summary.to_csv(path_summary, index=False)
table_p11_0_calibration_missing_components.to_csv(path_missing, index=False)

metadata = {
    "step": "P11.0",
    "title": "Survival Calibration Audit",
    "purpose": (
        "Audit currently available calibration artifacts and identify the next "
        "calibration-strengthening tasks for Group C using the current notebook convention."
    ),
    "has_model_level_calibration_bins_by_horizon": bool(has_model_level_bins),
    "has_model_level_calibration_summary": bool(has_model_level_summary),
    "has_model_level_brier_by_horizon": bool(has_model_level_brier),
    "has_model_level_support_by_horizon": bool(has_model_level_support),
    "has_benchmark_wide_calibration_table": bool(has_benchmark_wide_cal),
    "has_benchmark_wide_brier_table": bool(has_benchmark_wide_brier),
    "has_benchmark_wide_auc_table": bool(has_benchmark_wide_auc),
    "has_all_tuned_calibration_audit": bool(has_all_tuned_audit),
    "has_paper_summary_all_tuned_models": bool(has_paper_summary),
    "output_tables": [
        path_inventory.as_posix(),
        path_summary.as_posix(),
        path_missing.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nCalibration artifact inventory:")
display(table_p11_0_calibration_artifact_inventory)

print("\nCalibration audit summary:")
display(table_p11_0_calibration_audit_summary)

print("\nMissing / stronger calibration components:")
display(table_p11_0_calibration_missing_components)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_summary.resolve())
print("-", path_missing.resolve())
print("-", path_metadata.resolve())


# E3 — Horizon-wise Calibration Slope and Intercept Audit

In [ ]:
# ==============================================================
# E3 — Horizon-wise Calibration Slope and Intercept Audit
# REVISED VERSION (compatible with n and n_in_bin schemas)
# ==============================================================
# Purpose:
#   Estimate horizon-specific calibration intercept and slope
#   from current benchmark calibration bin artifacts.
#
# Method:
#   Weighted linear fit on:
#       logit(observed_event_rate) = intercept + slope * logit(mean_predicted_risk)
#   using calibration-bin summaries within each model_id x horizon_week.
#
# Main outputs:
#   - table_p11_1_calibration_slope_intercept_by_horizon.csv
#   - table_p11_1_calibration_slope_intercept_summary.csv
#   - metadata_p11_1_calibration_slope_intercept_audit.json
#
# Revision note:
#   This version supports both legacy reliability-bin schema using `n`
#   and current consolidated schema using `n_in_bin`.
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E3 — Horizon-wise Calibration Slope and Intercept Audit")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

reliability_all_path = OUT_TABLES / "table_calibration_reliability_bins_all_tuned_models.csv"
reliability_h20_path = OUT_TABLES / "table_calibration_reliability_bins_h20_all_tuned_models.csv"

if reliability_all_path.exists():
    rel_df = pd.read_csv(reliability_all_path)
    source_type = "reliability_bins_all_tuned_models"
elif reliability_h20_path.exists():
    rel_df = pd.read_csv(reliability_h20_path)
    source_type = "reliability_bins_h20_all_tuned_models"
else:
    raise FileNotFoundError(
        "No reliability-bin calibration artifact found. "
        "Expected table_calibration_reliability_bins_all_tuned_models.csv "
        "or table_calibration_reliability_bins_h20_all_tuned_models.csv."
    )

# ------------------------------
# 1) Schema checks
# ------------------------------
required_cols = [
    "horizon_week",
    "calibration_bin",
    "mean_predicted_risk",
    "observed_event_rate",
    "model_id",
    "model",
    "family",
]
missing_cols = [c for c in required_cols if c not in rel_df.columns]
if missing_cols:
    raise KeyError(f"Reliability-bin table is missing required columns: {missing_cols}")

# weight column: current schema prefers n_in_bin; legacy schema used n
if "n_in_bin" in rel_df.columns:
    weight_col = "n_in_bin"
elif "n" in rel_df.columns:
    weight_col = "n"
else:
    raise KeyError(
        "Reliability-bin table is missing the weight column. "
        "Expected either 'n_in_bin' (current schema) or 'n' (legacy schema)."
    )


### Funcao: clip_prob

Definicao isolada de clip_prob para reutilizacao nas etapas seguintes.


In [ ]:

# ------------------------------
# 2) Helpers
# ------------------------------
def clip_prob(x):
    x = pd.to_numeric(x, errors="coerce").astype(float)
    return np.clip(x, 1e-6, 1 - 1e-6)


### Funcao: weighted_mean

Definicao isolada de weighted_mean para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    den = np.sum(w)
    if den <= 0:
        return np.nan
    return np.sum(w * x) / den


### Funcao: weighted_var

Definicao isolada de weighted_var para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_var(x, w):
    mx = weighted_mean(x, w)
    return weighted_mean((x - mx) ** 2, w)


### Funcao: weighted_cov

Definicao isolada de weighted_cov para reutilizacao nas etapas seguintes.


In [ ]:

def weighted_cov(x, y, w):
    mx = weighted_mean(x, w)
    my = weighted_mean(y, w)
    return weighted_mean((x - mx) * (y - my), w)


### Funcao: fit_weighted_line

Definicao isolada de fit_weighted_line para reutilizacao nas etapas seguintes.


In [ ]:

def fit_weighted_line(x, y, w):
    var_x = weighted_var(x, w)
    if pd.isna(var_x) or var_x <= 0:
        return np.nan, np.nan
    slope = weighted_cov(x, y, w) / var_x
    intercept = weighted_mean(y, w) - slope * weighted_mean(x, w)
    return intercept, slope


In [ ]:

# ------------------------------
# 3) Prepare inputs
# ------------------------------
rel_df = rel_df.copy()
rel_df["horizon_week"] = pd.to_numeric(rel_df["horizon_week"], errors="coerce")
rel_df[weight_col] = pd.to_numeric(rel_df[weight_col], errors="coerce")

group_cols = ["model_id", "model", "family", "horizon_week"]
results = []

# ------------------------------
# 4) Estimate by model x horizon
# ------------------------------
for (model_id, model, family, horizon_week), g in rel_df.groupby(group_cols, dropna=False):
    pred = clip_prob(g["mean_predicted_risk"])
    obs = clip_prob(g["observed_event_rate"])
    w = pd.to_numeric(g[weight_col], errors="coerce").fillna(0).astype(float)

    valid_mask = (
        np.isfinite(pred) &
        np.isfinite(obs) &
        np.isfinite(w) &
        (w > 0)
    )

    pred = pred[valid_mask]
    obs = obs[valid_mask]
    w = w[valid_mask]

    if len(pred) < 2:
        results.append({
            "model_id": str(model_id),
            "model": str(model),
            "family": str(family),
            "horizon_week": int(horizon_week) if pd.notna(horizon_week) else np.nan,
            "calibration_intercept_logit": np.nan,
            "calibration_slope_logit": np.nan,
            "n_bins_used": int(len(pred)),
            "total_weight": float(np.sum(w)) if len(w) > 0 else 0.0,
            "source_type": source_type,
            "method_note": (
                "Insufficient valid bins for weighted logit-scale calibration fit."
            ),
        })
        continue

    x = np.log(pred / (1 - pred))
    y = np.log(obs / (1 - obs))

    intercept, slope = fit_weighted_line(x, y, w)

    results.append({
        "model_id": str(model_id),
        "model": str(model),
        "family": str(family),
        "horizon_week": int(horizon_week),
        "calibration_intercept_logit": intercept,
        "calibration_slope_logit": slope,
        "n_bins_used": int(len(pred)),
        "total_weight": float(np.sum(w)),
        "source_type": source_type,
        "method_note": (
            "Weighted linear fit on logit(observed_event_rate) vs "
            "logit(mean_predicted_risk) across calibration bins."
        ),
    })

if len(results) == 0:
    raise RuntimeError("No calibration slope/intercept estimates could be produced.")

# ------------------------------
# 5) Main table
# ------------------------------
table_p11_1_calibration_slope_intercept_by_horizon = (
    pd.DataFrame(results)
    .sort_values(["model_id", "horizon_week"])
    .reset_index(drop=True)
)

# ------------------------------
# 6) Summary table
# ------------------------------
summary_rows = []
for model_id, g in table_p11_1_calibration_slope_intercept_by_horizon.groupby("model_id", dropna=False):
    summary_rows.append({
        "model_id": model_id,
        "model": g["model"].iloc[0],
        "family": g["family"].iloc[0],
        "n_horizons_estimated": int(g["horizon_week"].nunique()),
        "mean_calibration_intercept_logit": pd.to_numeric(g["calibration_intercept_logit"], errors="coerce").mean(),
        "mean_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").mean(),
        "min_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").min(),
        "max_calibration_slope_logit": pd.to_numeric(g["calibration_slope_logit"], errors="coerce").max(),
        "source_type": g["source_type"].iloc[0],
    })

table_p11_1_calibration_slope_intercept_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("model_id")
    .reset_index(drop=True)
)

# ------------------------------
# 7) Save
# ------------------------------
path_main = OUT_TABLES / "table_p11_1_calibration_slope_intercept_by_horizon.csv"
path_summary = OUT_TABLES / "table_p11_1_calibration_slope_intercept_summary.csv"
path_metadata = OUT_METADATA / "metadata_p11_1_calibration_slope_intercept_audit.json"

table_p11_1_calibration_slope_intercept_by_horizon.to_csv(path_main, index=False)
table_p11_1_calibration_slope_intercept_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P11.1",
    "title": "Horizon-wise Calibration Slope and Intercept Audit",
    "purpose": (
        "Estimate horizon-specific approximate calibration intercept and slope "
        "from current reliability-bin calibration artifacts."
    ),
    "source_type": source_type,
    "weight_column_used": weight_col,
    "estimation_note": (
        "Approximate weighted linear fit on "
        "logit(observed_event_rate) vs logit(mean_predicted_risk) across calibration bins."
    ),
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------
# 8) Display
# ------------------------------
print(f"\nWeight column used: {weight_col}")

print("\nCalibration slope/intercept by horizon:")
display(table_p11_1_calibration_slope_intercept_by_horizon)

print("\nCalibration slope/intercept summary:")
display(table_p11_1_calibration_slope_intercept_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())


# E4 — Build Unified Calibration Strengthening Table

In [ ]:
# ==============================================================
# E4 — Build Unified Calibration Strengthening Table
# REVISED VERSION (aligned to current audit and P11.1 schemas)
# ==============================================================
# Purpose:
#   Consolidate current horizon-wise calibration evidence into
#   a single canonical table for paper/reviewer use.
#
# Inputs:
#   - table_calibration_audit_all_tuned_models.csv
#   - table_p11_1_calibration_slope_intercept_by_horizon.csv
#
# Main outputs:
#   - table_p11_2_calibration_strengthening_by_horizon.csv
#   - table_p11_2_calibration_strengthening_summary.csv
#   - metadata_p11_2_calibration_strengthening.json
#
# Revision note:
#   This version is aligned to the current benchmark schema, using:
#   - calibration_gap or weighted_absolute_calibration_gap_by_horizon
#   - support_n_at_horizon
#   - event_rate_at_horizon
#   - P11.1 logit-scale slope/intercept outputs
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E4 — Build Unified Calibration Strengthening Table")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve required sources
# --------------------------------------------------------------
path_audit = OUT_TABLES / "table_calibration_audit_all_tuned_models.csv"
path_slope = OUT_TABLES / "table_p11_1_calibration_slope_intercept_by_horizon.csv"

if not path_audit.exists():
    raise FileNotFoundError(f"Missing required file: {path_audit.resolve()}")
if not path_slope.exists():
    raise FileNotFoundError(f"Missing required file: {path_slope.resolve()}")

audit_df = pd.read_csv(path_audit)
slope_df = pd.read_csv(path_slope)

# --------------------------------------------------------------
# 2) Resolve current audit schema
# --------------------------------------------------------------
required_audit_base = ["model_id", "model", "family", "horizon_week"]
missing_base = [c for c in required_audit_base if c not in audit_df.columns]
if missing_base:
    raise KeyError(f"Audit table missing required identifier columns: {missing_base}")

if "weighted_absolute_calibration_gap_by_horizon" in audit_df.columns:
    audit_df = audit_df.copy()
    audit_df["calibration_gap_resolved"] = pd.to_numeric(
        audit_df["weighted_absolute_calibration_gap_by_horizon"], errors="coerce"
    )
elif "calibration_gap" in audit_df.columns:
    audit_df = audit_df.copy()
    audit_df["calibration_gap_resolved"] = pd.to_numeric(
        audit_df["calibration_gap"], errors="coerce"
    )
else:
    raise KeyError(
        "Audit table must contain either "
        "'weighted_absolute_calibration_gap_by_horizon' or 'calibration_gap'."
    )

# Support/event-rate current naming
if "support_n_at_horizon" in audit_df.columns:
    audit_df["support_n_resolved"] = pd.to_numeric(audit_df["support_n_at_horizon"], errors="coerce")
elif "n_evaluable_enrollments" in audit_df.columns:
    audit_df["support_n_resolved"] = pd.to_numeric(audit_df["n_evaluable_enrollments"], errors="coerce")
else:
    raise KeyError(
        "Audit table must contain either 'support_n_at_horizon' or 'n_evaluable_enrollments'."
    )

if "event_rate_at_horizon" in audit_df.columns:
    audit_df["event_rate_resolved"] = pd.to_numeric(audit_df["event_rate_at_horizon"], errors="coerce")
elif "event_rate_by_horizon" in audit_df.columns:
    audit_df["event_rate_resolved"] = pd.to_numeric(audit_df["event_rate_by_horizon"], errors="coerce")
else:
    raise KeyError(
        "Audit table must contain either 'event_rate_at_horizon' or 'event_rate_by_horizon'."
    )

# Optional fields
if "n_events_by_horizon" not in audit_df.columns:
    audit_df["n_events_by_horizon"] = np.nan

if "brier_at_horizon" not in audit_df.columns:
    audit_df["brier_at_horizon"] = np.nan

# Optional auxiliary proxy from current pipeline
if "abs_gap_proxy_from_bins" not in audit_df.columns:
    audit_df["abs_gap_proxy_from_bins"] = np.nan

# --------------------------------------------------------------
# 3) Resolve P11.1 schema
# --------------------------------------------------------------
required_slope_cols = [
    "model_id",
    "model",
    "family",
    "horizon_week",
    "calibration_intercept_logit",
    "calibration_slope_logit",
    "n_bins_used",
]
missing_slope = [c for c in required_slope_cols if c not in slope_df.columns]
if missing_slope:
    raise KeyError(f"Slope/intercept table missing required columns: {missing_slope}")

slope_df = slope_df.copy()
slope_df["horizon_week"] = pd.to_numeric(slope_df["horizon_week"], errors="coerce")
slope_df["calibration_intercept_logit"] = pd.to_numeric(
    slope_df["calibration_intercept_logit"], errors="coerce"
)
slope_df["calibration_slope_logit"] = pd.to_numeric(
    slope_df["calibration_slope_logit"], errors="coerce"
)
slope_df["n_bins_used"] = pd.to_numeric(slope_df["n_bins_used"], errors="coerce")
if "total_weight" in slope_df.columns:
    slope_df["total_weight"] = pd.to_numeric(slope_df["total_weight"], errors="coerce")
else:
    slope_df["total_weight"] = np.nan

# --------------------------------------------------------------
# 4) Prepare canonical merge inputs
# --------------------------------------------------------------
audit_core = audit_df.copy()
audit_core["horizon_week"] = pd.to_numeric(audit_core["horizon_week"], errors="coerce")

# Keep one row per model x horizon
audit_keep_cols = [
    "model_id",
    "model",
    "family",
    "horizon_week",
    "calibration_gap_resolved",
    "support_n_resolved",
    "n_events_by_horizon",
    "event_rate_resolved",
    "brier_at_horizon",
    "abs_gap_proxy_from_bins",
]
audit_core = (
    audit_core[audit_keep_cols]
    .drop_duplicates(subset=["model_id", "model", "family", "horizon_week"])
    .copy()
)

slope_core = (
    slope_df[
        [
            "model_id",
            "model",
            "family",
            "horizon_week",
            "calibration_intercept_logit",
            "calibration_slope_logit",
            "n_bins_used",
            "total_weight",
            "source_type",
            "method_note",
        ]
    ]
    .drop_duplicates(subset=["model_id", "model", "family", "horizon_week"])
    .copy()
)

# --------------------------------------------------------------
# 5) Merge into unified strengthening table
# --------------------------------------------------------------
table_p11_2_calibration_strengthening_by_horizon = audit_core.merge(
    slope_core,
    on=["model_id", "model", "family", "horizon_week"],
    how="left"
)

table_p11_2_calibration_strengthening_by_horizon = table_p11_2_calibration_strengthening_by_horizon.rename(
    columns={
        "calibration_gap_resolved": "calibration_gap",
        "support_n_resolved": "support_n_at_horizon",
        "event_rate_resolved": "event_rate_at_horizon",
        "calibration_intercept_logit": "approx_calibration_intercept_by_horizon",
        "calibration_slope_logit": "approx_calibration_slope_by_horizon",
    }
)


### Funcao: primary_gap_reading

Definicao isolada de primary_gap_reading para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 6) Diagnostic readings / flags
# --------------------------------------------------------------
def primary_gap_reading(gap):
    if pd.isna(gap):
        return "missing"
    if gap <= 0.04:
        return "strong"
    if gap <= 0.08:
        return "acceptable"
    return "weaker"


### Funcao: slope_reading_logit

Definicao isolada de slope_reading_logit para reutilizacao nas etapas seguintes.


In [ ]:

def slope_reading_logit(s):
    if pd.isna(s):
        return "missing"
    if 0.90 <= s <= 1.10:
        return "near_ideal"
    if 0.75 <= s < 0.90:
        return "mild_compression"
    if 1.10 < s <= 1.25:
        return "mild_overdispersion"
    if s < 0.75:
        return "strong_compression"
    return "strong_overdispersion"


### Funcao: intercept_magnitude_reading

Definicao isolada de intercept_magnitude_reading para reutilizacao nas etapas seguintes.


In [ ]:

def intercept_magnitude_reading(i):
    if pd.isna(i):
        return "missing"
    if abs(i) <= 0.15:
        return "near_ideal"
    if 0.15 < abs(i) <= 0.40:
        return "moderate_shift"
    return "large_shift"


### Funcao: intercept_direction

Definicao isolada de intercept_direction para reutilizacao nas etapas seguintes.


In [ ]:

def intercept_direction(i):
    if pd.isna(i):
        return "missing"
    if abs(i) <= 0.15:
        return "no_clear_bias"
    if i < 0:
        return "systematic_overprediction"
    return "systematic_underprediction"


### Funcao: strengthening_status

Definicao isolada de strengthening_status para reutilizacao nas etapas seguintes.


In [ ]:

def strengthening_status(intercept, slope):
    problems = 0
    if pd.notna(intercept) and abs(intercept) > 0.40:
        problems += 1
    elif pd.notna(intercept) and abs(intercept) > 0.15:
        problems += 0.5

    if pd.notna(slope) and (slope < 0.75 or slope > 1.25):
        problems += 1
    elif pd.notna(slope) and ((0.75 <= slope < 0.90) or (1.10 < slope <= 1.25)):
        problems += 0.5

    if problems == 0:
        return "supportive"
    if problems <= 1:
        return "caution"
    return "problematic"


### Funcao: combined_flag

Definicao isolada de combined_flag para reutilizacao nas etapas seguintes.


In [ ]:

def combined_flag(gap, intercept, slope):
    primary = primary_gap_reading(gap)
    strength = strengthening_status(intercept, slope)

    if primary == "missing":
        return "insufficient_information"
    if primary == "strong" and strength == "supportive":
        return "strong"
    if primary == "strong" and strength in ["caution", "problematic"]:
        return "acceptable_with_caveats"
    if primary == "acceptable" and strength in ["supportive", "caution"]:
        return "acceptable_with_caveats"
    return "weaker_or_problematic"


In [ ]:

table_p11_2_calibration_strengthening_by_horizon["primary_gap_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["calibration_gap"].apply(primary_gap_reading)
)
table_p11_2_calibration_strengthening_by_horizon["slope_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_slope_by_horizon"].apply(slope_reading_logit)
)
table_p11_2_calibration_strengthening_by_horizon["intercept_magnitude_reading"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_intercept_by_horizon"].apply(intercept_magnitude_reading)
)
table_p11_2_calibration_strengthening_by_horizon["intercept_direction"] = (
    table_p11_2_calibration_strengthening_by_horizon["approx_calibration_intercept_by_horizon"].apply(intercept_direction)
)
table_p11_2_calibration_strengthening_by_horizon["strengthening_status"] = (
    table_p11_2_calibration_strengthening_by_horizon.apply(
        lambda row: strengthening_status(
            row["approx_calibration_intercept_by_horizon"],
            row["approx_calibration_slope_by_horizon"],
        ),
        axis=1,
    )
)
table_p11_2_calibration_strengthening_by_horizon["calibration_flag"] = (
    table_p11_2_calibration_strengthening_by_horizon.apply(
        lambda row: combined_flag(
            row["calibration_gap"],
            row["approx_calibration_intercept_by_horizon"],
            row["approx_calibration_slope_by_horizon"],
        ),
        axis=1,
    )
)

table_p11_2_calibration_strengthening_by_horizon = (
    table_p11_2_calibration_strengthening_by_horizon
    .sort_values(["horizon_week", "calibration_gap", "brier_at_horizon"], ascending=[True, True, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 7) Summary table
# --------------------------------------------------------------
summary_rows = []
for model_id, g in table_p11_2_calibration_strengthening_by_horizon.groupby("model_id", dropna=False):
    share_strong = (g["calibration_flag"] == "strong").mean()
    share_ok = (g["calibration_flag"].isin(["strong", "acceptable_with_caveats"])).mean()

    if share_strong >= 2/3:
        overall_reading = "strong"
    elif share_ok >= 2/3:
        overall_reading = "acceptable_with_caveats"
    else:
        overall_reading = "weaker_or_problematic"

    summary_rows.append({
        "model_id": model_id,
        "model": g["model"].iloc[0],
        "family": g["family"].iloc[0],
        "n_horizons_estimated": int(g["horizon_week"].nunique()),
        "mean_calibration_gap": pd.to_numeric(g["calibration_gap"], errors="coerce").mean(),
        "mean_brier_at_horizon": pd.to_numeric(g["brier_at_horizon"], errors="coerce").mean(),
        "mean_event_rate_at_horizon": pd.to_numeric(g["event_rate_at_horizon"], errors="coerce").mean(),
        "mean_support_n_at_horizon": pd.to_numeric(g["support_n_at_horizon"], errors="coerce").mean(),
        "mean_calibration_intercept_logit": pd.to_numeric(g["approx_calibration_intercept_by_horizon"], errors="coerce").mean(),
        "mean_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").mean(),
        "min_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").min(),
        "max_calibration_slope_logit": pd.to_numeric(g["approx_calibration_slope_by_horizon"], errors="coerce").max(),
        "best_horizon_by_calibration_gap": (
            g.loc[pd.to_numeric(g["calibration_gap"], errors="coerce").idxmin(), "horizon_week"]
            if g["calibration_gap"].notna().any() else np.nan
        ),
        "overall_calibration_reading": overall_reading,
        "source_type": g["source_type"].dropna().iloc[0] if g["source_type"].notna().any() else np.nan,
    })

table_p11_2_calibration_strengthening_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["mean_calibration_gap", "mean_brier_at_horizon"], ascending=[True, True])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 8) Save outputs
# --------------------------------------------------------------
path_main = OUT_TABLES / "table_p11_2_calibration_strengthening_by_horizon.csv"
path_summary = OUT_TABLES / "table_p11_2_calibration_strengthening_summary.csv"
path_metadata = OUT_METADATA / "metadata_p11_2_calibration_strengthening.json"

table_p11_2_calibration_strengthening_by_horizon.to_csv(path_main, index=False)
table_p11_2_calibration_strengthening_summary.to_csv(path_summary, index=False)

metadata = {
    "step": "P11.2",
    "title": "Build Unified Calibration Strengthening Table",
    "purpose": (
        "Consolidate calibration gap, support, event rate, "
        "and slope/intercept diagnostics into a single canonical table."
    ),
    "inputs": [
        path_audit.as_posix(),
        path_slope.as_posix(),
    ],
    "resolved_audit_fields": {
        "calibration_gap": "weighted_absolute_calibration_gap_by_horizon or calibration_gap",
        "support_n_at_horizon": "support_n_at_horizon or n_evaluable_enrollments",
        "event_rate_at_horizon": "event_rate_at_horizon or event_rate_by_horizon",
    },
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 9) Display
# --------------------------------------------------------------
print("\nUnified calibration strengthening table:")
display(table_p11_2_calibration_strengthening_by_horizon)

print("\nCalibration strengthening summary:")
display(table_p11_2_calibration_strengthening_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_metadata.resolve())


# E5 — Sensitivity Design Audit

In [ ]:
# ==============================================================
# E5 — Sensitivity Design Audit
# ==============================================================
# Purpose:
#   Audit current sensitivity-design coverage in the notebook
#   and identify what is already present vs. what is still
#   missing for Group D.
#
# Main outputs:
#   - table_p12_0_sensitivity_artifact_inventory.csv
#   - table_p12_0_sensitivity_design_summary.csv
#   - table_p12_0_sensitivity_missing_components.csv
#   - metadata_p12_0_sensitivity_design_audit.json
# ==============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

print("=" * 70)
print("E5 — Sensitivity Design Audit")
print("=" * 70)

OUT_BASE = Path("outputs_benchmark_survival")
OUT_TABLES = OUT_BASE / "tables"
OUT_METADATA = OUT_BASE / "metadata"
OUT_FIGURES = OUT_BASE / "figures"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_METADATA.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Inventory potentially relevant sensitivity artifacts
# --------------------------------------------------------------
candidate_files = []

tokens = [
    "sensitivity",
    "ablation",
    "window",
    "horizon",
    "support_by_horizon",
    "brier_by_horizon",
    "risk_auc_by_horizon",
    "calibration_by_horizon",
    "benchmark_",
    "variant",
]

for folder, kind in [
    (OUT_TABLES, "table"),
    (OUT_FIGURES, "figure"),
    (OUT_METADATA, "metadata"),
]:
    if folder.exists():
        for p in sorted(folder.iterdir()):
            name_l = p.name.lower()
            if any(tok in name_l for tok in tokens):
                candidate_files.append({
                    "artifact_name": p.name,
                    "artifact_path": str(p.resolve()),
                    "artifact_type": kind,
                    "exists": True,
                    "size_bytes": p.stat().st_size if p.exists() else np.nan,
                })

table_p12_0_sensitivity_artifact_inventory = pd.DataFrame(candidate_files)

if table_p12_0_sensitivity_artifact_inventory.empty:
    table_p12_0_sensitivity_artifact_inventory = pd.DataFrame([{
        "artifact_name": "(none found)",
        "artifact_path": "",
        "artifact_type": "none",
        "exists": False,
        "size_bytes": np.nan,
    }])

table_names = set(
    table_p12_0_sensitivity_artifact_inventory.loc[
        table_p12_0_sensitivity_artifact_inventory["artifact_type"] == "table",
        "artifact_name"
    ].astype(str).tolist()
)


### Funcao: exists_exact

Definicao isolada de exists_exact para reutilizacao nas etapas seguintes.


In [ ]:

# --------------------------------------------------------------
# 2) Coverage checks for reviewer-requested sensitivity dimensions
# --------------------------------------------------------------
def exists_exact(name: str) -> bool:
    return name in table_names


### Funcao: exists_contains

Definicao isolada de exists_contains para reutilizacao nas etapas seguintes.


In [ ]:

def exists_contains(substr: str) -> bool:
    return any(substr in name for name in table_names)


In [ ]:

has_ablation_continuous = exists_exact("table_ablation_continuous_calibration_by_horizon.csv")
has_ablation_discrete = exists_exact("table_ablation_discrete_calibration_by_horizon.csv")

has_benchmark_auc_horizons = exists_exact("table_benchmark_risk_auc_by_horizon_wide.csv")
has_benchmark_brier_horizons = exists_exact("table_benchmark_brier_by_horizon_wide.csv")
has_benchmark_calib_horizons = exists_exact("table_benchmark_calibration_by_horizon_wide.csv")

has_window_features_logic = True  # by design, because P12 exists in notebook structure
has_horizon_evaluation = has_benchmark_auc_horizons or has_benchmark_brier_horizons or has_benchmark_calib_horizons

# More specific missing reviewer items
has_explicit_window_length_sensitivity = exists_contains("first_2") or exists_contains("first_4") or exists_contains("first_6")
has_explicit_discretization_scheme_sensitivity = exists_contains("km") or exists_contains("quantile") or exists_contains("equidistant")
has_explicit_horizon_choice_sensitivity = has_horizon_evaluation

# --------------------------------------------------------------
# 3) Build corrected design summary
# --------------------------------------------------------------
summary_rows = [
    {
        "sensitivity_dimension": "ablation_currently_present",
        "available": bool(has_ablation_continuous or has_ablation_discrete),
        "notes": "Ablation artifacts are already present in the notebook outputs."
    },
    {
        "sensitivity_dimension": "horizon_wise_benchmark_outputs",
        "available": bool(has_horizon_evaluation),
        "notes": "Benchmark-wide horizon outputs already exist for AUC/Brier/calibration."
    },
    {
        "sensitivity_dimension": "window_feature_design_present",
        "available": bool(has_window_features_logic),
        "notes": "Window-based design is already part of the notebook structure."
    },
    {
        "sensitivity_dimension": "explicit_window_length_sensitivity",
        "available": bool(has_explicit_window_length_sensitivity),
        "notes": "Evidence of multiple early-window lengths should be confirmed structurally, not inferred only from names."
    },
    {
        "sensitivity_dimension": "explicit_discretization_scheme_sensitivity",
        "available": bool(has_explicit_discretization_scheme_sensitivity),
        "notes": "No clear evidence yet of a canonical sensitivity comparison across discretization schemes."
    },
    {
        "sensitivity_dimension": "explicit_horizon_choice_sensitivity",
        "available": bool(has_explicit_horizon_choice_sensitivity),
        "notes": "Horizon-wise outputs exist, but stress-test framing may still need to be made explicit."
    },
]

table_p12_0_sensitivity_design_summary = pd.DataFrame(summary_rows)

# --------------------------------------------------------------
# 4) Missing / next components
# --------------------------------------------------------------
missing_components = [
    {
        "component": "early_window_length_sensitivity",
        "status": (
            "partially_present_but_not_yet_canonical"
            if has_explicit_window_length_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned the choice of early-window length.",
        "recommended_next_step": "Create a canonical sensitivity comparison across at least 2/4/6-week early-window designs.",
        "current_evidence_note": (
            "Some window-specific naming may already exist, but not yet as a canonical sensitivity table."
            if has_explicit_window_length_sensitivity else
            "No canonical early-window sensitivity artifact identified."
        ),
    },
    {
        "component": "discretization_scheme_sensitivity",
        "status": (
            "partially_present_but_not_yet_canonical"
            if has_explicit_discretization_scheme_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned discretization choices.",
        "recommended_next_step": "Create a canonical comparison across discretization schemes or document why one harmonized scheme is retained.",
        "current_evidence_note": (
            "Some discretization-related naming was detected, but not yet as a canonical stress-test artifact."
            if has_explicit_discretization_scheme_sensitivity else
            "No canonical discretization sensitivity artifact identified."
        ),
    },
    {
        "component": "horizon_choice_stress_test",
        "status": (
            "already_partially_present"
            if has_explicit_horizon_choice_sensitivity else
            "still_missing"
        ),
        "why_it_matters": "Reviewer explicitly questioned whether 10/20/30 was stress-tested.",
        "recommended_next_step": "Reframe existing horizon-wise benchmark outputs as an explicit horizon-choice sensitivity analysis.",
        "current_evidence_note": (
            "Horizon-wise outputs already exist, but may need explicit sensitivity framing."
            if has_explicit_horizon_choice_sensitivity else
            "No canonical horizon stress-test artifact identified."
        ),
    },
    {
        "component": "canonical_sensitivity_summary_table",
        "status": "still_missing",
        "why_it_matters": "Needed for clear reporting in the paper and reviewer response.",
        "recommended_next_step": "Build one unified table summarizing sensitivity results across the selected dimensions.",
        "current_evidence_note": "No canonical unified sensitivity table identified yet.",
    },
]

table_p12_0_sensitivity_missing_components = pd.DataFrame(missing_components)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_inventory = OUT_TABLES / "table_p12_0_sensitivity_artifact_inventory.csv"
path_summary = OUT_TABLES / "table_p12_0_sensitivity_design_summary.csv"
path_missing = OUT_TABLES / "table_p12_0_sensitivity_missing_components.csv"
path_metadata = OUT_METADATA / "metadata_p12_0_sensitivity_design_audit.json"

table_p12_0_sensitivity_artifact_inventory.to_csv(path_inventory, index=False)
table_p12_0_sensitivity_design_summary.to_csv(path_summary, index=False)
table_p12_0_sensitivity_missing_components.to_csv(path_missing, index=False)

metadata = {
    "step": "P12.0",
    "title": "Sensitivity Design Audit",
    "purpose": (
        "Audit current sensitivity-design coverage and identify the next "
        "structural tasks for Group D."
    ),
    "has_ablation_artifacts": bool(has_ablation_continuous or has_ablation_discrete),
    "has_horizon_wise_benchmark_outputs": bool(has_horizon_evaluation),
    "has_window_feature_design_present": bool(has_window_features_logic),
    "output_tables": [
        path_inventory.as_posix(),
        path_summary.as_posix(),
        path_missing.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity artifact inventory:")
display(table_p12_0_sensitivity_artifact_inventory)

print("\nSensitivity design summary:")
display(table_p12_0_sensitivity_design_summary)

print("\nMissing / next sensitivity components:")
display(table_p12_0_sensitivity_missing_components)

print("\nSaved:")
print("-", path_inventory.resolve())
print("-", path_summary.resolve())
print("-", path_missing.resolve())
print("-", path_metadata.resolve())


# E6 — Horizon-Choice Stress-Test Framing

### Funcao: wide_to_long_metric

Definicao isolada de wide_to_long_metric para reutilizacao nas etapas seguintes.


In [ ]:
# ==============================================================
# E6 — Horizon-Choice Stress-Test Framing
# ==============================================================
# Purpose:
#   Reframe existing benchmark horizon-wise outputs as an explicit
#   horizon-choice sensitivity analysis for Group D.
#
# Main outputs:
#   - table_p12_3_horizon_choice_stress_test.csv
#   - table_p12_3_horizon_choice_summary.csv
#   - table_p12_3_horizon_choice_registry.csv
#   - metadata_p12_3_horizon_choice_stress_test.json
# ==============================================================

print("\n" + "=" * 70)
print("E6 — Horizon-Choice Stress-Test Framing")
print("=" * 70)

required_names = ["TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = Path(OUTPUT_DIR)
OUT_METADATA = OUT_BASE / "metadata"
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve source files
# --------------------------------------------------------------
path_auc = TABLES_DIR / "table_benchmark_risk_auc_by_horizon_wide.csv"
path_brier = TABLES_DIR / "table_benchmark_brier_by_horizon_wide.csv"
path_calib = TABLES_DIR / "table_benchmark_calibration_by_horizon_wide.csv"
path_registry = TABLES_DIR / "table_benchmark_model_registry.csv"

required_files = [path_auc, path_brier, path_calib]
missing_files = [str(p.resolve()) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required benchmark horizon files for P12.3:\n- "
        + "\n- ".join(missing_files)
    )

auc_df = pd.read_csv(path_auc)
brier_df = pd.read_csv(path_brier)
calib_df = pd.read_csv(path_calib)
registry_df = pd.read_csv(path_registry) if path_registry.exists() else None

# --------------------------------------------------------------
# 2) Helper: wide-to-long by h10/h20/h30
# --------------------------------------------------------------
def wide_to_long_metric(df: pd.DataFrame, metric_name: str) -> pd.DataFrame:
    required_base = ["model_id", "display_name", "family", "variant"]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise KeyError(f"{metric_name} table is missing required columns: {missing}")

    horizon_cols = [c for c in df.columns if c.lower().startswith("h")]
    if not horizon_cols:
        raise ValueError(f"{metric_name} table has no horizon columns.")

    out = df.melt(
        id_vars=required_base,
        value_vars=horizon_cols,
        var_name="horizon_label",
        value_name=metric_name,
    ).copy()

    out["horizon_week"] = (
        out["horizon_label"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .astype(float)
    )
    return out.drop(columns=["horizon_label"])

auc_long = wide_to_long_metric(auc_df, "risk_auc")
brier_long = wide_to_long_metric(brier_df, "brier")
calib_long = wide_to_long_metric(calib_df, "calibration_gap")

# --------------------------------------------------------------
# 3) Merge canonical horizon-stress-test table
# --------------------------------------------------------------
merge_cols = ["model_id", "display_name", "family", "variant", "horizon_week"]

stress_df = auc_long.merge(
    brier_long,
    on=merge_cols,
    how="outer",
    validate="one_to_one",
).merge(
    calib_long,
    on=merge_cols,
    how="outer",
    validate="one_to_one",
)

stress_df = stress_df.sort_values(["model_id", "horizon_week"]).reset_index(drop=True)

# optional registry enrichment
if registry_df is not None and "model_id" in registry_df.columns:
    keep_cols = [c for c in ["model_id", "benchmark_arm", "representation_type", "update_regime", "model_class"] if c in registry_df.columns]
    if keep_cols:
        stress_df = stress_df.merge(
            registry_df[keep_cols].drop_duplicates(subset=["model_id"]),
            on="model_id",
            how="left",
        )

# stress-test framing columns
stress_df["horizon_choice_role"] = stress_df["horizon_week"].map({
    10.0: "short_horizon",
    20.0: "mid_horizon",
    30.0: "long_horizon",
}).fillna("other_horizon")

stress_df["horizon_choice_in_scope"] = stress_df["horizon_week"].isin([10, 20, 30])

# --------------------------------------------------------------
# 4) Registry table
# --------------------------------------------------------------
table_p12_3_horizon_choice_registry = (
    stress_df[
        [
            c for c in stress_df.columns
            if c in [
                "model_id", "display_name", "family", "variant",
                "benchmark_arm", "representation_type", "update_regime", "model_class"
            ]
        ]
    ]
    .drop_duplicates()
    .sort_values(["family", "model_id"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Summary table
# --------------------------------------------------------------
summary_rows = []
for model_id, g in stress_df.groupby("model_id", dropna=False):
    summary_rows.append({
        "model_id": model_id,
        "display_name": g["display_name"].iloc[0] if "display_name" in g.columns else None,
        "family": g["family"].iloc[0] if "family" in g.columns else None,
        "variant": g["variant"].iloc[0] if "variant" in g.columns else None,
        "n_horizons_in_scope": int(pd.to_numeric(g["horizon_choice_in_scope"], errors="coerce").fillna(False).sum()),
        "min_horizon_week": pd.to_numeric(g["horizon_week"], errors="coerce").min(),
        "max_horizon_week": pd.to_numeric(g["horizon_week"], errors="coerce").max(),
        "mean_risk_auc": pd.to_numeric(g["risk_auc"], errors="coerce").mean(),
        "mean_brier": pd.to_numeric(g["brier"], errors="coerce").mean(),
        "mean_calibration_gap": pd.to_numeric(g["calibration_gap"], errors="coerce").mean(),
        "risk_auc_range": (
            pd.to_numeric(g["risk_auc"], errors="coerce").max()
            - pd.to_numeric(g["risk_auc"], errors="coerce").min()
        ),
        "brier_range": (
            pd.to_numeric(g["brier"], errors="coerce").max()
            - pd.to_numeric(g["brier"], errors="coerce").min()
        ),
        "calibration_gap_range": (
            pd.to_numeric(g["calibration_gap"], errors="coerce").max()
            - pd.to_numeric(g["calibration_gap"], errors="coerce").min()
        ),
        "horizon_stress_test_status": (
            "in_scope_complete"
            if set(pd.to_numeric(g["horizon_week"], errors="coerce").dropna().astype(int).tolist()) >= {10, 20, 30}
            else "partial"
        ),
    })

table_p12_3_horizon_choice_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["mean_risk_auc", "mean_calibration_gap"], ascending=[False, True])
    .reset_index(drop=True)
)

table_p12_3_horizon_choice_stress_test = stress_df.copy()

# --------------------------------------------------------------
# 6) Save outputs
# --------------------------------------------------------------
path_main = TABLES_DIR / "table_p12_3_horizon_choice_stress_test.csv"
path_summary = TABLES_DIR / "table_p12_3_horizon_choice_summary.csv"
path_registry_out = TABLES_DIR / "table_p12_3_horizon_choice_registry.csv"
path_metadata = OUT_METADATA / "metadata_p12_3_horizon_choice_stress_test.json"

table_p12_3_horizon_choice_stress_test.to_csv(path_main, index=False)
table_p12_3_horizon_choice_summary.to_csv(path_summary, index=False)
table_p12_3_horizon_choice_registry.to_csv(path_registry_out, index=False)

metadata = {
    "step": "P12.3",
    "title": "Horizon-Choice Stress-Test Framing",
    "purpose": (
        "Reframe existing benchmark horizon-wise outputs as an explicit "
        "horizon-choice sensitivity analysis for Group D."
    ),
    "horizons_in_scope": [10, 20, 30],
    "source_tables": [
        path_auc.as_posix(),
        path_brier.as_posix(),
        path_calib.as_posix(),
    ],
    "output_tables": [
        path_main.as_posix(),
        path_summary.as_posix(),
        path_registry_out.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 7) Display
# --------------------------------------------------------------
print("\nHorizon-choice registry:")
display(table_p12_3_horizon_choice_registry)

print("\nHorizon-choice stress-test table:")
display(table_p12_3_horizon_choice_stress_test)

print("\nHorizon-choice summary:")
display(table_p12_3_horizon_choice_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_summary.resolve())
print("-", path_registry_out.resolve())
print("-", path_metadata.resolve())

# E7 — Build Unified Sensitivity Summary Table

In [ ]:
# ==============================================================
# E7 — Build Unified Sensitivity Summary Table
# FULL REWRITTEN VERSION WITH EXPLICIT GRANULARITY
# ==============================================================
# Purpose:
#   Consolidate Group D sensitivity evidence into a canonical
#   summary table for reviewer response and paper revision,
#   while avoiding ambiguous duplication across family-level
#   and model-level rows.
#
# Main outputs:
#   - table_p12_4_sensitivity_unified_summary.csv
#   - table_p12_4_sensitivity_component_registry.csv
#   - table_p12_4_sensitivity_family_summary.csv
#   - metadata_p12_4_sensitivity_unified_summary.json
# ==============================================================

print("\n" + "=" * 70)
print("E7 — Build Unified Sensitivity Summary Table")
print("=" * 70)

required_names = ["TABLES_DIR", "OUTPUT_DIR"]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required objects from previous cells: "
        + ", ".join(missing_names)
        + ". Please run prior setup cells first."
    )

import json
from pathlib import Path
import pandas as pd
import numpy as np

OUT_BASE = Path(OUTPUT_DIR)
OUT_METADATA = OUT_BASE / "metadata"
OUT_METADATA.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------
# 1) Resolve required inputs
# --------------------------------------------------------------
path_window_registry = TABLES_DIR / "table_p12_2_window_sensitivity_registry.csv"
path_window_summary = TABLES_DIR / "table_p12_2_window_sensitivity_summary.csv"
path_horizon_summary = TABLES_DIR / "table_p12_3_horizon_choice_summary.csv"
path_horizon_stress = TABLES_DIR / "table_p12_3_horizon_choice_stress_test.csv"

required_files = [
    path_window_registry,
    path_window_summary,
    path_horizon_summary,
    path_horizon_stress,
]
missing_files = [str(p.resolve()) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required Group D files for P12.4:\n- " + "\n- ".join(missing_files)
    )

window_registry_df = pd.read_csv(path_window_registry)
window_summary_df = pd.read_csv(path_window_summary)
horizon_summary_df = pd.read_csv(path_horizon_summary)
horizon_stress_df = pd.read_csv(path_horizon_stress)

# Optional ablation artifacts
optional_ablation_files = {
    "table_ablation_leaderboard_all_tuned.csv": TABLES_DIR / "table_ablation_leaderboard_all_tuned.csv",
    "table_ablation_summary_by_model.csv": TABLES_DIR / "table_ablation_summary_by_model.csv",
    "table_ablation_scenario_comparison.csv": TABLES_DIR / "table_ablation_scenario_comparison.csv",
    "table_ablation_continuous_primary_metrics.csv": TABLES_DIR / "table_ablation_continuous_primary_metrics.csv",
    "table_ablation_discrete_primary_metrics.csv": TABLES_DIR / "table_ablation_discrete_primary_metrics.csv",
}
ablation_presence = {name: path.exists() for name, path in optional_ablation_files.items()}

# --------------------------------------------------------------
# 2) Component registry
# --------------------------------------------------------------
component_rows = []

# 2A) Early-window components -> family-level
for _, r in window_summary_df.iterrows():
    component_rows.append({
        "entity_level": "family",
        "entity_id": r.get("family"),
        "display_name": None,
        "family": r.get("family"),
        "variant": None,
        "sensitivity_component": "early_window_length",
        "scope_label": "2_4_6_week_design",
        "status": (
            "materialized_and_ready"
            if bool(r.get("all_windows_ready_for_model_training", False))
            else "partially_ready"
        ),
        "n_expected_units": r.get("n_windows_expected"),
        "n_materialized_units": r.get("n_windows_materialized"),
        "n_ready_units": r.get("n_windows_ready_for_model_training"),
        "reference_value": "4-week main window",
        "notes": (
            f"Materialized windows: {r.get('window_list_materialized', '')}. "
            "Structural design ready for explicit window-length sensitivity."
        ),
    })

# 2B) Horizon components -> model-level
for _, r in horizon_summary_df.iterrows():
    component_rows.append({
        "entity_level": "model",
        "entity_id": r.get("model_id"),
        "display_name": r.get("display_name"),
        "family": r.get("family"),
        "variant": r.get("variant"),
        "sensitivity_component": "horizon_choice",
        "scope_label": "10_20_30_horizon_stress_test",
        "status": r.get("horizon_stress_test_status"),
        "n_expected_units": 3,
        "n_materialized_units": r.get("n_horizons_in_scope"),
        "n_ready_units": r.get("n_horizons_in_scope"),
        "reference_value": "10/20/30 benchmark horizons",
        "notes": (
            f"Mean AUC={r.get('mean_risk_auc'):.6f}, "
            f"Mean Brier={r.get('mean_brier'):.6f}, "
            f"Mean calibration gap={r.get('mean_calibration_gap'):.6f}."
            if pd.notna(r.get("mean_risk_auc", np.nan))
            else "Horizon-wise benchmark outputs consolidated."
        ),
    })

# 2C) Ablation component -> global-level
component_rows.append({
    "entity_level": "global",
    "entity_id": "global",
    "display_name": None,
    "family": "global",
    "variant": None,
    "sensitivity_component": "ablation",
    "scope_label": "existing_ablation_artifacts",
    "status": "present" if any(ablation_presence.values()) else "not_detected",
    "n_expected_units": len(optional_ablation_files),
    "n_materialized_units": int(sum(ablation_presence.values())),
    "n_ready_units": int(sum(ablation_presence.values())),
    "reference_value": "existing ablation outputs",
    "notes": (
        "Detected ablation artifacts: "
        + ", ".join([name for name, present in ablation_presence.items() if present])
        if any(ablation_presence.values()) else
        "No optional ablation artifacts detected."
    ),
})

table_p12_4_sensitivity_component_registry = (
    pd.DataFrame(component_rows)
    .sort_values(["entity_level", "family", "entity_id", "sensitivity_component"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 3) Unified summary table
# --------------------------------------------------------------
summary_rows = []

# 3A) family-level early-window summary
for _, r in window_summary_df.iterrows():
    summary_rows.append({
        "entity_level": "family",
        "entity_id": r.get("family"),
        "display_name": None,
        "family": r.get("family"),
        "variant": None,
        "summary_layer": "early_window_length",
        "reference_design": "4-week main window",
        "scope": "2/4/6 weeks",
        "status": (
            "ready"
            if bool(r.get("all_windows_ready_for_model_training", False))
            else "partial"
        ),
        "n_units_in_scope": r.get("n_windows_expected"),
        "n_units_materialized": r.get("n_windows_materialized"),
        "key_signal_1": r.get("window_list_materialized"),
        "key_signal_2": "all_windows_ready_for_model_training=" + str(r.get("all_windows_ready_for_model_training")),
        "notes": "Canonical early-window sensitivity design is structurally available.",
    })

# 3B) model-level horizon-choice summary
for _, r in horizon_summary_df.iterrows():
    summary_rows.append({
        "entity_level": "model",
        "entity_id": r.get("model_id"),
        "display_name": r.get("display_name"),
        "family": r.get("family"),
        "variant": r.get("variant"),
        "summary_layer": "horizon_choice",
        "reference_design": "10/20/30 benchmark horizons",
        "scope": "10/20/30",
        "status": r.get("horizon_stress_test_status"),
        "n_units_in_scope": 3,
        "n_units_materialized": r.get("n_horizons_in_scope"),
        "key_signal_1": r.get("mean_risk_auc"),
        "key_signal_2": r.get("mean_calibration_gap"),
        "notes": "Existing horizon-wise outputs are explicitly framed here as a horizon-choice stress test.",
    })

# 3C) global ablation summary
summary_rows.append({
    "entity_level": "global",
    "entity_id": "global",
    "display_name": None,
    "family": "global",
    "variant": None,
    "summary_layer": "ablation",
    "reference_design": "existing ablation protocol",
    "scope": "current ablation outputs",
    "status": "present" if any(ablation_presence.values()) else "not_detected",
    "n_units_in_scope": len(optional_ablation_files),
    "n_units_materialized": int(sum(ablation_presence.values())),
    "key_signal_1": int(sum(ablation_presence.values())),
    "key_signal_2": len(optional_ablation_files),
    "notes": (
        "Ablation evidence already exists and can be cited as part of Group D."
        if any(ablation_presence.values()) else
        "No ablation artifacts detected."
    ),
})

table_p12_4_sensitivity_unified_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["entity_level", "family", "entity_id", "summary_layer"])
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 4) Family-level summary
# --------------------------------------------------------------
family_rows = []
for family, g in table_p12_4_sensitivity_unified_summary.groupby("family", dropna=False):
    family_rows.append({
        "family": family,
        "n_rows": int(len(g)),
        "n_unique_entities": int(g["entity_id"].astype(str).nunique()),
        "n_summary_layers": int(g["summary_layer"].astype(str).nunique()),
        "n_ready_or_present_rows": int(g["status"].astype(str).isin(["ready", "present", "in_scope_complete"]).sum()),
        "all_rows_nonempty": bool(len(g) > 0),
        "entity_levels_present": ", ".join(sorted(g["entity_level"].astype(str).unique().tolist())),
        "summary_layers_present": ", ".join(sorted(g["summary_layer"].astype(str).unique().tolist())),
    })

table_p12_4_sensitivity_family_summary = (
    pd.DataFrame(family_rows)
    .sort_values("family")
    .reset_index(drop=True)
)

# --------------------------------------------------------------
# 5) Save outputs
# --------------------------------------------------------------
path_main = TABLES_DIR / "table_p12_4_sensitivity_unified_summary.csv"
path_registry = TABLES_DIR / "table_p12_4_sensitivity_component_registry.csv"
path_family = TABLES_DIR / "table_p12_4_sensitivity_family_summary.csv"
path_metadata = OUT_METADATA / "metadata_p12_4_sensitivity_unified_summary.json"

table_p12_4_sensitivity_unified_summary.to_csv(path_main, index=False)
table_p12_4_sensitivity_component_registry.to_csv(path_registry, index=False)
table_p12_4_sensitivity_family_summary.to_csv(path_family, index=False)

metadata = {
    "step": "P12.4",
    "title": "Build Unified Sensitivity Summary Table",
    "purpose": (
        "Consolidate Group D sensitivity evidence across early-window design, "
        "horizon-choice stress test, and existing ablation artifacts, with explicit granularity."
    ),
    "granularity_rule": {
        "early_window_length": "family-level",
        "horizon_choice": "model-level",
        "ablation": "global-level"
    },
    "input_tables": [
        path_window_registry.as_posix(),
        path_window_summary.as_posix(),
        path_horizon_summary.as_posix(),
        path_horizon_stress.as_posix(),
    ],
    "optional_ablation_artifacts_detected": [
        name for name, present in ablation_presence.items() if present
    ],
    "output_tables": [
        path_main.as_posix(),
        path_registry.as_posix(),
        path_family.as_posix(),
    ],
}
with open(path_metadata, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# --------------------------------------------------------------
# 6) Display
# --------------------------------------------------------------
print("\nSensitivity component registry:")
display(table_p12_4_sensitivity_component_registry)

print("\nUnified sensitivity summary:")
display(table_p12_4_sensitivity_unified_summary)

print("\nSensitivity family summary:")
display(table_p12_4_sensitivity_family_summary)

print("\nSaved:")
print("-", path_main.resolve())
print("-", path_registry.resolve())
print("-", path_family.resolve())
print("-", path_metadata.resolve())

In [ ]:
# RZ - DuckDB explicit shutdown (run this as the last cell)
if "con" in globals() and con is not None:
    try:
        con.execute("CHECKPOINT")
    except Exception as _checkpoint_err:
        print(f"Warning during DuckDB CHECKPOINT: {_checkpoint_err}")

    try:
        con.close()
        print("DuckDB connection closed.")
    except Exception as _close_err:
        print(f"Warning while closing DuckDB connection: {_close_err}")
    finally:
        con = None
else:
    print("No active DuckDB connection found in variable 'con'.")